# Transaction preparation with PySpark

This notebook imports transaction data, adjusts its schema, applies business filters, builds category totals, and saves results in CSV and Parquet formats.

## Part A: Start Spark and import the data

A Spark session is created and the transaction source is read into a DataFrame.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round

spark = SparkSession.builder \
    .appName("Transaction_Pipeline") \
    .getOrCreate()

print("Spark Session initiated.")


In [ ]:
transactions_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/source.csv")

print("Raw Ingested Transactions Preview:")
transactions_df.show()


## Part B: Select fields and apply initial filters

Required identifiers are selected and the target product category is isolated.

In [ ]:
transactions_df.select("product_id", "price").show()


In [ ]:
print("Filtering for Electronics category:")
transactions_df.filter(col("category") == "Electronics").show()


## Part C: Update fields and calculate tax-inclusive prices

Column names and types are standardized before an 18% tax is added to the price.

In [ ]:
# Replace the legacy name field with the preferred label
transformed_df = transactions_df.withColumnRenamed("old_name", "new_name")
transformed_df.show()


In [ ]:
# Convert the price field to double precision
transformed_df = transformed_df.withColumn("price", col("price").cast("double"))
transformed_df.printSchema()


In [ ]:
# Derive the tax-inclusive price, rounded to two decimal places
transformed_df = transformed_df.withColumn("final_price", round(col("base_price") * 1.18, 2))
transformed_df.show()


## Part D: Apply combined business conditions

The notebook selects expensive completed orders, handles missing user identifiers, and retrieves the required regional or priority records.

In [ ]:
# Find completed orders whose amount is above 1,000
transformed_df.filter(
    (col("status") == "Completed") & (col("amount") > 1000)
).show()


In [ ]:
# Remove transactions without a user identifier
transformed_df.filter(col("user_id").isNotNull()).show()


In [ ]:
# Select North-region items or records marked High priority
transformed_df.filter(
    (col("region") == "North") | (col("priority") == "High")
).show()


## Part E: Aggregate and save the results

Record counts are calculated by category, then the resulting datasets are saved as CSV and Parquet files.

In [ ]:
# Count records within each product category
category_summary = transformed_df.groupBy("category").count()
category_summary.show()


In [ ]:
# Write the grouped results to CSV
category_summary.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/csv_output")


In [ ]:
# Write the detailed data to Parquet
transformed_df.write \
    .mode("overwrite") \
    .parquet("output/parquet_output")


In [ ]:
# Reopen the Parquet output as a verification step
verification_df = spark.read.parquet("output/parquet_output")
print("Verifying Parquet Output:")
verification_df.show()

# Stop the Spark session
spark.stop()
